In [19]:
import pandas as pd
import os

# ============================
# Paths
# ============================
super_path = "/Users/judycheng/Desktop/supercharger in washington state.xls"
output_path = "/Users/judycheng/Desktop/supercharger_by_county_summary.xlsx"

# ============================
# Read file
# ============================
df = pd.read_excel(super_path)

# ============================
# Filter Washington
# ============================
df_wa = df[df["State"] == "Washington"]

# ============================
# Count all charge *points*  
# (1 row = 1 charger point)
# ============================
counts = (
    df_wa.groupby("County")["County"]
    .count()
    .reset_index(name="Supercharger_Count")
)

# ============================
# Clean "County" suffix
# ============================
counts["County"] = counts["County"].str.replace(" County", "", regex=False)

# Sort descending
counts = counts.sort_values("Supercharger_Count", ascending=False)

# ============================
# Export to Excel
# ============================
counts.to_excel(output_path, index=False)

print("Done! File exported to:", output_path)


Done! File exported to: /Users/judycheng/Desktop/supercharger_by_county_summary.xlsx


In [21]:
import pandas as pd

# =========================
# Paths
# =========================
residents_path = "/Users/judycheng/Desktop/Population 2024 age 25 to 59.xlsx"
super_summary_path = "/Users/judycheng/Desktop/supercharger_by_county_summary.xlsx"
ev_path = "/Users/judycheng/Desktop/coordinates_output.xlsm"

output_forecast_path = (
    "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"
)

# =========================
# 1) Read Residents (Population) file
# =========================
residents_df = pd.read_excel(residents_path)

# Clean County names
residents_df["County"] = (
    residents_df["County"].astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

# ❗ Explicitly use the "total" column as Pop_2024
if "total" not in residents_df.columns:
    raise ValueError("The residents file does not contain a 'total' column.")

pop_df = residents_df[["County", "total"]].rename(columns={"total": "Pop_2024"})

# =========================
# 2) Read Supercharger summary (points per county)
# =========================
super_df = pd.read_excel(super_summary_path)
super_df["County"] = super_df["County"].astype(str).str.strip()

super_df = super_df.rename(columns={"Supercharger_Count": "Superchargers_2024"})
super_df = super_df[["County", "Superchargers_2024"]]

# =========================
# 3) Read EV registration file (VIN count)
# =========================
ev_df = pd.read_excel(ev_path)
ev_df["County"] = (
    ev_df["County"].astype(str)
    .str.replace(" County", "", regex=False)
    .str.strip()
)

ev_counts = (
    ev_df.groupby("County")["County"]
    .count()
    .reset_index(name="EVs_2024")
)

# =========================
# 4) Build 2024 baseline
# =========================
base = pop_df.merge(super_df, on="County", how="left")
base = base.merge(ev_counts, on="County", how="left")

base["Superchargers_2024"] = base["Superchargers_2024"].fillna(0).astype(int)
base["EVs_2024"] = base["EVs_2024"].fillna(0).astype(int)

base["Adoption_2024"] = (
    base["EVs_2024"] / base["Pop_2024"]
).fillna(0)

# Order columns
base = base[["County", "Pop_2024", "Superchargers_2024", "EVs_2024", "Adoption_2024"]]

# =========================
# 5) Add 2025–2050 empty forecast columns
# =========================
for year in range(2025, 2051):
    base[f"Superchargers_{year}"] = pd.NA
    base[f"EVs_{year}"] = pd.NA
    base[f"Adoption_{year}"] = pd.NA

# Sort alphabetically
base = base.sort_values("County").reset_index(drop=True)

# =========================
# 6) Export to Excel
# =========================
base.to_excel(output_forecast_path, index=False)

print("✅ Forecast baseline file created:")
print(output_forecast_path)


✅ Forecast baseline file created:
/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx


In [22]:
import pandas as pd

# ========================================================
# FILE PATHS
# ========================================================
base_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"

king_path   = "/Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx"
pierce_path = "/Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx"
kitsap_path = "/Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx"
chelan_path = "/Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx"

output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"

YEARS = list(range(2025, 2051))


# ========================================================
# 1. LOAD BASELINE FILE
# ========================================================
df = pd.read_excel(base_path)
df["County"] = df["County"].astype(str).str.strip()


# ========================================================
# 2. LOAD MC TEMPLATE FUNCTION
# ========================================================
def load_mc(path):
    mc = pd.read_excel(path, sheet_name="Forecast")
    mc = mc.rename(columns={mc.columns[0]: "Year"})
    mc = mc.set_index("Year")

    needed = ["Forecast_Chargers", "Forecast_EVs_P50", "Forecast_Adoption_P50"]
    for c in needed:
        if c not in mc.columns:
            raise ValueError(f"Missing column {c} in template: {path}")

    return mc[needed]


# Load all 4 MC templates
king_mc   = load_mc(king_path)
pierce_mc = load_mc(pierce_path)
kitsap_mc = load_mc(kitsap_path)
chelan_mc = load_mc(chelan_path)


# ========================================================
# 3. SELECT TEMPLATE BASED ON POPULATION
# ========================================================
def select_template(pop):
    if pop > 1_000_000:
        return king_mc
    elif pop > 130_000:
        return pierce_mc
    elif pop > 34_000:
        return kitsap_mc
    else:
        return chelan_mc


# ========================================================
# 4. FILL IN FORECAST VALUES FOR 2025–2050
# ========================================================
for idx, row in df.iterrows():
    pop = row["Pop_2024"]
    template = select_template(pop)

    for year in YEARS:

        # Read adoption and chargers from MC template
        adoption = template.loc[year, "Forecast_Adoption_P50"]
        chargers = template.loc[year, "Forecast_Chargers"]

        # EVs = Adoption × Pop_2024
        evs = adoption * pop

        df.at[idx, f"Adoption_{year}"]      = adoption
        df.at[idx, f"Superchargers_{year}"] = chargers
        df.at[idx, f"EVs_{year}"]           = evs


# Optional: convert numeric
for year in YEARS:
    df[f"Superchargers_{year}"] = pd.to_numeric(df[f"Superchargers_{year}"], errors="coerce")
    df[f"EVs_{year}"]           = pd.to_numeric(df[f"EVs_{year}"], errors="coerce")
    df[f"Adoption_{year}"]      = pd.to_numeric(df[f"Adoption_{year}"], errors="coerce")


# ========================================================
# 5. EXPORT FINAL FILE
# ========================================================
df.to_excel(output_path, index=False)

print("✅ Monte Carlo forecast (2025–2050) successfully filled.")
print("📄 Output saved to:")
print(output_path)


✅ Monte Carlo forecast (2025–2050) successfully filled.
📄 Output saved to:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx


In [2]:
import pandas as pd

# ========================================================
# FILE PATHS
# ========================================================
base_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_baseline_2024_to_2050.xlsx"

king_path   = "/Users/judycheng/Desktop/king_county_ev_projection_mc_monotonic.xlsx"
pierce_path = "/Users/judycheng/Desktop/pierce_county_ev_projection_mc_monotonic.xlsx"
kitsap_path = "/Users/judycheng/Desktop/kitsap_county_ev_projection_mc_monotonic.xlsx"
chelan_path = "/Users/judycheng/Desktop/chelan_county_ev_projection_mc_monotonic.xlsx"

output_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"

YEARS = list(range(2025, 2051))


# ========================================================
# 1. LOAD BASELINE FILE
# ========================================================
df = pd.read_excel(base_path)
df["County"] = df["County"].astype(str).str.strip()


# ========================================================
# 2. MC TEMPLATE LOADER
# ========================================================
def load_mc(path):
    mc = pd.read_excel(path, sheet_name="Forecast")
    mc = mc.rename(columns={mc.columns[0]: "Year"})
    mc = mc.set_index("Year")

    needed = ["Forecast_Chargers", "Forecast_EVs_P50", "Forecast_Adoption_P50"]
    for c in needed:
        if c not in mc.columns:
            raise ValueError(f"Missing column {c} in template: {path}")

    return mc[needed]


# Load templates
king_mc   = load_mc(king_path)
pierce_mc = load_mc(pierce_path)
kitsap_mc = load_mc(kitsap_path)
chelan_mc = load_mc(chelan_path)


# ========================================================
# 3. SELECT TEMPLATE BASED ON POPULATION
# ========================================================
def select_template(pop):
    if pop > 1_000_000:
        return king_mc
    elif pop > 130_000:
        return pierce_mc
    elif pop > 34_000:
        return kitsap_mc
    else:
        return chelan_mc


# ========================================================
# 4. FILL FORECAST (2025–2050)
# ========================================================
for idx, row in df.iterrows():
    pop = row["Pop_2024"]
    template = select_template(pop)

    for year in YEARS:
        adoption = template.loc[year, "Forecast_Adoption_P50"]
        chargers = template.loc[year, "Forecast_Chargers"]
        evs = adoption * pop

        df.at[idx, f"Adoption_{year}"]      = adoption
        df.at[idx, f"Superchargers_{year}"] = chargers
        df.at[idx, f"EVs_{year}"]           = evs


# Convert to numeric
for year in YEARS:
    df[f"Superchargers_{year}"] = pd.to_numeric(df[f"Superchargers_{year}"], errors="coerce")
    df[f"EVs_{year}"]           = pd.to_numeric(df[f"EVs_{year}"], errors="coerce")
    df[f"Adoption_{year}"]      = pd.to_numeric(df[f"Adoption_{year}"], errors="coerce")


# ========================================================
# 5. ADD TOTAL ROW AT BOTTOM
# ========================================================

total = {}

# County label
total["County"] = "TOTAL"

# Pop_2024 sum
total["Pop_2024"] = df["Pop_2024"].sum()

# Sum EV and Charger columns & weighted average for Adoption
for col in df.columns:
    if col.startswith("EVs_") or col.startswith("Superchargers_"):
        total[col] = df[col].sum(skipna=True)

    elif col.startswith("Adoption_"):
        year = col.split("_")[1]

        # Weighted average:
        # sum(pop * adoption) / sum(pop)
        weighted = (df["Pop_2024"] * df[col]).sum() / df["Pop_2024"].sum()
        total[col] = weighted


# Append row
df = pd.concat([df, pd.DataFrame([total])], ignore_index=True)


# ========================================================
# 6. EXPORT FINAL FILE
# ========================================================
df.to_excel(output_path, index=False)

print("✅ Monte Carlo forecast (2025–2050) successfully filled.")
print("➕ TOTAL row added.")
print("📄 Output saved to:")
print(output_path)


✅ Monte Carlo forecast (2025–2050) successfully filled.
➕ TOTAL row added.
📄 Output saved to:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from matplotlib.ticker import PercentFormatter

# ======================================================
# FILE PATHS
# ======================================================
excel_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"
desktop = os.path.join(os.path.expanduser("~"), "Desktop")

chart1_png = os.path.join(desktop, "chart1_adoption_evs_vs_year.png")
chart2_png = os.path.join(desktop, "chart2_chargers_vs_year.png")
chart3_png = os.path.join(desktop, "chart3_adopt_evs_vs_chargers.png")

# ======================================================
# 1. LOAD FINAL FILE & TOTAL ROW
# ======================================================
df = pd.read_excel(excel_path)
total = df[df["County"] == "TOTAL"].iloc[0]

years = list(range(2024, 2051))

# Series from TOTAL row
adopt = [total[f"Adoption_{y}"] for y in years]
evs = [total[f"EVs_{y}"] for y in years]
# 🚩 FIX: use TOTAL superchargers directly (already the stock level by year)
chargers = [total.get(f"Superchargers_{y}", 0) for y in years]

# For sanity check (optional)
print(f"Max chargers (should be ~1259): {max(chargers)}")

# ======================================================
# 2. CREATE CHARTS (dots only) + SAVE AS PNG
# ======================================================

# -------- Chart 1 --------
# Adoption rate & EVs vs Year
plt.figure(figsize=(10, 6))
plt.scatter(years, evs, s=60, label="EV Registrations")

ax1 = plt.gca()
ax2 = ax1.twinx()
ax2.scatter(years, adopt, s=60, marker="x", label="Adoption Rate")

ax1.set_title("Statewide Adoption Rate & EV Registrations vs Year (2024–2050)")
ax1.set_xlabel("Year")
ax1.set_ylabel("EV Registrations")
ax2.set_ylabel("Adoption Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

ax1.grid(True)

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="best")

plt.savefig(chart1_png, dpi=300, bbox_inches="tight")
plt.close()


# -------- Chart 2 --------
# Total Superchargers vs Year (NO extra cumsum)
plt.figure(figsize=(10, 6))
plt.scatter(years, chargers, s=60)

plt.title("Total Superchargers vs Year (2024–2050)")
plt.xlabel("Year")
plt.ylabel("Total Superchargers")
plt.grid(True)

plt.savefig(chart2_png, dpi=300, bbox_inches="tight")
plt.close()


# -------- Chart 3 --------
# EV Registrations (left Y) & Adoption Rate (right Y) vs Total Superchargers (X)
plt.figure(figsize=(10, 6))

x = chargers

ax1 = plt.gca()
ax1.scatter(x, evs, s=60, label="EV Registrations")
ax1.set_xlabel("Total Superchargers")
ax1.set_ylabel("EV Registrations")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.scatter(x, adopt, s=60, marker="x", label="Adoption Rate")
ax2.set_ylabel("Adoption Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

plt.title("Adoption Rate & EV Registrations vs Total Superchargers")

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="best")

plt.savefig(chart3_png, dpi=300, bbox_inches="tight")
plt.close()

print("✅ PNG charts created on Desktop.")


# ======================================================
# 3. INSERT CHARTS INTO 'Charts' SHEET IN EXCEL
# ======================================================
wb = load_workbook(excel_path)

# Remove old Charts tab if exists
if "Charts" in wb.sheetnames:
    ws = wb["Charts"]
    wb.remove(ws)

ws = wb.create_sheet("Charts")

img1 = XLImage(chart1_png)
img2 = XLImage(chart2_png)
img3 = XLImage(chart3_png)

ws.add_image(img1, "A1")
ws.add_image(img2, "A35")
ws.add_image(img3, "A70")

wb.save(excel_path)

print("📊 Charts inserted into Excel tab: 'Charts'")
print("📄 File updated:")
print(excel_path)


Max chargers (should be ~1259): 1259
✅ PNG charts created on Desktop.
📊 Charts inserted into Excel tab: 'Charts'
📄 File updated:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx


In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import r2_score

# ======================================================
# FILE PATHS
# ======================================================
excel_path = "/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx"
desktop = os.path.join(os.path.expanduser("~"), "Desktop")

chart1_png = os.path.join(desktop, "chart1_adoption_evs_vs_year.png")
chart2_png = os.path.join(desktop, "chart2_chargers_vs_year.png")
chart3_png = os.path.join(desktop, "chart3_adopt_evs_vs_chargers.png")

# ======================================================
# 1. LOAD FINAL FILE & TOTAL ROW
# ======================================================
df = pd.read_excel(excel_path)
total = df[df["County"] == "TOTAL"].iloc[0]

years = list(range(2024, 2051))

# Series from TOTAL row
adopt = [total[f"Adoption_{y}"] for y in years]          # adoption in decimal (0–1)
evs = [total[f"EVs_{y}"] for y in years]
chargers = [total.get(f"Superchargers_{y}", 0) for y in years]

print(f"Max chargers (should be ~1259): {max(chargers)}")

# ======================================================
# 2. BEST-FIT CURVE (FORCED y(0)=0)
# ======================================================
def best_fit_through_zero(x, y, label):
    x_arr = np.array(x, dtype=float)
    y_arr = np.array(y, dtype=float)

    results = {}

    for deg in (1, 2, 3):

        # Design matrix WITHOUT intercept:
        # deg=1 -> [x]
        # deg=2 -> [x, x^2]
        # deg=3 -> [x, x^2, x^3]
        X = np.vstack([(x_arr ** p) for p in range(1, deg + 1)]).T

        coef, _, _, _ = np.linalg.lstsq(X, y_arr, rcond=None)
        y_pred = X @ coef
        r2 = r2_score(y_arr, y_pred)
        results[deg] = (coef, r2)

    best_deg = max(results, key=lambda d: results[d][1])
    coef_best, r2_best = results[best_deg]

    # Build readable formula
    pieces = []
    for i, a in enumerate(coef_best, start=1):
        if i == 1:
            pieces.append(f"{a:.6f} * x")
        else:
            pieces.append(f"{a:.6f} * x^{i}")

    formula_str = " + ".join(pieces)

    model_name = (
        "Linear through origin" if best_deg == 1 else
        "Quadratic through origin" if best_deg == 2 else
        "Cubic through origin"
    )

    desc = f"{label} best-fit ({model_name}, R²={r2_best:.4f}): y = {formula_str}"

    print("\n📈", desc)
    print("   Constrained: y(0) = 0 (no negative intercept).")

    return best_deg, coef_best, desc, r2_best

# ======================================================
# 3. CREATE CHARTS (dots only) + SAVE AS PNG
# ======================================================

# -------- Chart 1 --------
plt.figure(figsize=(10, 6))
ax1 = plt.gca()
ax1.scatter(years, evs, s=60, label="EV Registrations")

ax2 = ax1.twinx()
ax2.scatter(years, adopt, s=60, marker="x", label="Adoption Rate")

ax1.set_title("Statewide Adoption Rate & EV Registrations vs Year (2024–2050)")
ax1.set_xlabel("Year")
ax1.set_ylabel("EV Registrations")
ax2.set_ylabel("Adoption Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))
ax1.grid(True)

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="best")

plt.savefig(chart1_png, dpi=300, bbox_inches="tight")
plt.close()


# -------- Chart 2 --------
plt.figure(figsize=(10, 6))
plt.scatter(years, chargers, s=60)
plt.title("Total Superchargers vs Year (2024–2050)")
plt.xlabel("Year")
plt.ylabel("Total Superchargers")
plt.grid(True)
plt.savefig(chart2_png, dpi=300, bbox_inches="tight")
plt.close()


# -------- Chart 3 --------
plt.figure(figsize=(10, 6))

x = chargers

ax1 = plt.gca()
ax1.scatter(x, evs, s=60, label="EV Registrations")
ax1.set_xlabel("Total Superchargers")
ax1.set_ylabel("EV Registrations")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.scatter(x, adopt, s=60, marker="x", label="Adoption Rate")
ax2.set_ylabel("Adoption Rate")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

plt.title("Adoption Rate & EV Registrations vs Total Superchargers")

# ----- BEST-FIT (CONSTRAINED y(0)=0) -----
deg_evs, coef_evs, desc_evs, r2_evs = best_fit_through_zero(x, evs, "EV Registrations")
deg_adopt, coef_adopt, desc_adopt, r2_adopt = best_fit_through_zero(x, adopt, "Adoption Rate")

# Smooth curve
x_line = np.linspace(min(x), max(x), 300)

# EV curve
X_line_evs = np.vstack([(x_line ** p) for p in range(1, deg_evs + 1)]).T
y_evs_fit = X_line_evs @ coef_evs
ax1.plot(x_line, y_evs_fit, linestyle="--", label="EV Best Fit")

# Adoption curve
X_line_adopt = np.vstack([(x_line ** p) for p in range(1, deg_adopt + 1)]).T
y_adopt_fit = X_line_adopt @ coef_adopt
ax2.plot(x_line, y_adopt_fit, linestyle="--", label="Adoption Best Fit")

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="best")

plt.savefig(chart3_png, dpi=300, bbox_inches="tight")
plt.close()

print("✅ PNG charts created on Desktop.")

# ======================================================
# 4. INSERT CHARTS + FORMULAS INTO 'Charts' SHEET
# ======================================================
wb = load_workbook(excel_path)

if "Charts" in wb.sheetnames:
    wb.remove(wb["Charts"])

ws = wb.create_sheet("Charts")

# Insert images
ws.add_image(XLImage(chart1_png), "A1")
ws.add_image(XLImage(chart2_png), "A35")
ws.add_image(XLImage(chart3_png), "A70")

# Add best-fit formulas
ws["A105"] = "Best-fit curves for Chart 3 (y vs Total Superchargers x):"
ws["A106"] = desc_evs
ws["A107"] = desc_adopt
ws["A109"] = "Note: Adoption rate y is in decimal form (0–1)."

wb.save(excel_path)

print("📊 Charts and best-fit formulas written to Excel tab: 'Charts'")
print("📄 File updated:")
print(excel_path)


Max chargers (should be ~1259): 1259

📈 EV Registrations best-fit (Cubic through origin, R²=0.9641): y = 3773.495357 * x + -3.578990 * x^2 + 0.002106 * x^3
   Constrained: y(0) = 0 (no negative intercept).

📈 Adoption Rate best-fit (Cubic through origin, R²=0.9641): y = 0.000999 * x + -0.000001 * x^2 + 0.000000 * x^3
   Constrained: y(0) = 0 (no negative intercept).
✅ PNG charts created on Desktop.
📊 Charts and best-fit formulas written to Excel tab: 'Charts'
📄 File updated:
/Users/judycheng/Desktop/wa_county_ev_forecast_filled_mc.xlsx
